In [ ]:
"""
NDVI Visualization — Etosha National Park
==========================================
Reads matrix CSV + shapefile, reprojects to metre-based AEQD coords
centred on the park, interpolates NDVI onto a regular grid, and plots.
 
 Dependencies:
     pip install pandas numpy matplotlib pyproj geopandas fiona shapely scipy
     """
      
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pyproj import Transformer
import geopandas as gpd
from scipy.interpolate import griddata

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH   = "ndvi.csv"
SHP_PATH   = "WDPA_WDOECM_Apr2026_Public_884_shp-polygons.shp"
OUTPUT_PNG = "etosha_ndvi_map.png"
NDVI_SCALE = 1000.0   # raw values are NDVI * 1000
GRID_RES   = 5        # interpolation grid resolution in km
# ─────────────────────────────────────────────────────────────────────────────

# 1. Load CSV → long format
df_raw = pd.read_csv(CSV_PATH, header=0, index_col=0)
df_raw.index   = df_raw.index.astype(float)
df_raw.columns = df_raw.columns.astype(float)
    
records = []
for lat in df_raw.index:
for lon in df_raw.columns:
        records.append({"lat": lat, "lon": lon,
                                "ndvi": df_raw.loc[lat, lon] / NDVI_SCALE})
                                df = pd.DataFrame(records)
                                
                                # 2. Load shapefile
                                os.environ["SHAPE_RESTORE_SHX"] = "YES"
                                park = gpd.read_file(SHP_PATH)
                                park = park.set_crs("EPSG:4326") if park.crs is None else park.to_crs("EPSG:4326")
                                
                                # 3. AEQD projection centred on park centroid
                                centroid   = park.geometry.union_all().centroid
                                center_lon, center_lat = centroid.x, centroid.y
                                print(f"Centroid: lon={center_lon:.4f}  lat={center_lat:.4f}")
                                
                                proj_str = (f"+proj=aeqd +lat_0={center_lat} +lon_0={center_lon} "
                                            "+datum=WGS84 +units=m +no_defs")
                                                
                                                transformer = Transformer.from_crs("EPSG:4326", proj_str, always_xy=True)
                                                
                                                # 4. Reproject NDVI points → km
                                                x_m, y_m = transformer.transform(df["lon"].values, df["lat"].values)
                                                df["x_km"] = x_m / 1000
                                                df["y_km"] = y_m / 1000
                                                
                                                # 5. Reproject park polygon
                                                park_proj = park.to_crs(proj_str)
                                                
                                                # 6. Interpolate onto regular grid
                                                xi = np.arange(df.x_km.min() - 5, df.x_km.max() + 5, GRID_RES)
                                                yi = np.arange(df.y_km.min() - 5, df.y_km.max() + 5, GRID_RES)
                                                XI, YI = np.meshgrid(xi, yi)
                                                ZI = griddata((df.x_km, df.y_km), df.ndvi, (XI, YI), method="linear")
                                                    
                                                    # 7. Plot
                                                    fig, ax = plt.subplots(figsize=(14, 8), facecolor="#0d1117")
                                                    ax.set_facecolor("#0d1117")
                                                    
                                                    cmap = mcolors.LinearSegmentedColormap.from_list("ndvi", ["#ffffcc", "#006400"])
                                                    norm = mcolors.Normalize(vmin=0, vmax=1)
                                                    
                                                    ax.pcolormesh(XI, YI, ZI, cmap=cmap, norm=norm, shading="auto", zorder=1)
                                                    
                                                    # Park boundary
                                                    for geom in park_proj.geometry:
                                                        xs, ys = geom.exterior.xy
                                                            ax.plot([x / 1000 for x in xs], [y / 1000 for y in ys],
                                                                        color="black", linewidth=2.0, zorder=4, label="Park boundary")
                                                                            
                                                                            # Origin
                                                                            ax.plot(0, 0, "+", color="black", markersize=14,
                                                                                    markeredgewidth=2.5, zorder=6, label="Centroid (0, 0)")
                                                                                    
                                                                                    ax.grid(True, linestyle="--", alpha=0.2, color="white", zorder=0)
                                                                                    ax.set_xlabel("Easting from centroid (km)",  fontsize=11, color="#cccccc")
                                                                                    ax.set_ylabel("Northing from centroid (km)", fontsize=11, color="#cccccc")
                                                                                    ax.tick_params(colors="#aaaaaa", labelsize=9)
                                                                                    for spine in ax.spines.values():
                                                                                        spine.set_edgecolor("#444444")
                                                                                        
                                                                                        ax.set_title(
                                                                                            "NDVI — Etosha National Park\n"
                                                                                                "Azimuthal Equidistant · Origin = park centroid",
                                                                                                    fontsize=13, fontweight="bold", color="white", pad=16
                                                                                                    )
                                                                                                    
                                                                                                    sm   = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
                                                                                                    cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.025)
                                                                                                    cbar.set_label("NDVI  (0 = bare/salt pan  →  1 = dense vegetation)",
                                                                                                                    fontsize=10, color="#cccccc")
                                                                                                                    cbar.ax.yaxis.set_tick_params(color="#aaaaaa")
                                                                                                                    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#aaaaaa")
                                                                                                                    
                                                                                                                    # Deduplicate legend entries
                                                                                                                    handles, labels = ax.get_legend_handles_labels()
                                                                                                                    seen = {}
                                                                                                                    for h, l in zip(handles, labels):
                                                                                                                        if l not in seen:
                                                                                                                                seen[l] = h
                                                                                                                                ax.legend(seen.values(), seen.keys(), loc="upper right", fontsize=9,
                                                                                                                                            facecolor="#1e2433", edgecolor="#555555", labelcolor="white")
                                                                                                                                            
                                                                                                                                            ax.set_aspect("equal")
                                                                                                                                            plt.tight_layout()
                                                                                                                                            plt.savefig(OUTPUT_PNG, dpi=160, bbox_inches="tight",
                                                                                                                                                        facecolor=fig.get_facecolor())
                                                                                                                                                        print(f"Saved → {OUTPUT_PNG}")
                                                                                                                                                        plt.show()"""

IndentationError: unexpected indent (3394755362.py, line 33)